# NMF Topic Model — England k=5

**Corpus:** 3,931 articles post-cleaning (3,943 raw → cleaned by removing GOV.UK footer phrases, FFT newsletter boilerplate, and articles <200 chars)

**Sources:** SchoolsWeek (~69%), GOV.UK, FFT, EPI, Nuffield, FED

**Model:** NMF, k=5, max_features=3000, min_df=3, init=nndsvd, max_iter=1000

**Purpose:** One of four England model variants trained for the contestability demonstration. `nmf_eng_30` (production baseline), `nmf_eng_15`, `nmf_eng_5`, and `nmf_eng_30_nm` (no media) show how specification choices — topic count and source composition — change what the model finds. See `docs/internal/current/technical/contestability_example.md`.

**Key finding:** At k=5, source concentration collapses — 3 of 5 topics are effectively single-source (100% Gov, 100% SchoolsWeek, 86% SchoolsWeek). The model stops discovering policy themes and starts discovering which outlet publishes what. This is the strongest evidence that k is not a technical parameter but a specification choice that determines whether the output reflects a policy debate or a media landscape.


# 0. Imports 

In [1]:
import sys
sys.path.insert(0, "/workspaces/AM1_topic_modelling")

import logging
import pandas as pd
import numpy as np
from pathlib import Path

logging.basicConfig(level=logging.INFO)

from model_pipeline.training.s02_cleaning import run_cleaning
from model_pipeline.training.s03_spacy_processing import run_spacy_processing
from model_pipeline.training.s04_vectorisation import run_vectorisation, build_vectorizer
from model_pipeline.training.s05_nmf_training import run_nmf_training, get_top_words_per_topic
from model_pipeline.training.s06_topic_allocation import TOPIC_NAMES

import logging
logging.getLogger("gensim").setLevel(logging.WARNING)

from sklearn.decomposition import NMF
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

INFO:model_pipeline.training.s06_topic_allocation:Loaded 5 topic names from llm_topic_review.json


# 1. Load England training data

In [2]:
preprocessed_path = Path("/workspaces/AM1_topic_modelling/data/training/eng_training_preprocessed.parquet")

if preprocessed_path.exists():
    print("Loading preprocessed data (skipping cleaning + spaCy)...")
    df = pd.read_parquet(preprocessed_path)
    SKIP_PREPROCESSING = True
else:
    csv_path = Path("/workspaces/AM1_topic_modelling/data/training/eng_training.csv")
    df = pd.read_csv(csv_path)
    SKIP_PREPROCESSING = False

print(f"Loaded: {df.shape}")
print(f"Skip preprocessing: {SKIP_PREPROCESSING}")

Loading preprocessed data (skipping cleaning + spaCy)...
Loaded: (3943, 18)
Skip preprocessing: True


# 2. Prepare text column

The Supabase schema has `title` and `text` as separate columns. The pipeline expects a combined `text` column. Also rename `article_date` → `date` to match pipeline expectations.

In [3]:
if not SKIP_PREPROCESSING:
    # Combine title + text (same as s01_data_loader)
    df["text"] = df["title"].fillna("") + "\n\n" + df["text"].fillna("")
    df["date"] = pd.to_datetime(df["article_date"], errors="coerce")
    print(f"Text combined. Sample length: {df['text'].str.len().describe()}")
else:
    print("Skipped — using preprocessed data")

Skipped — using preprocessed data


## 3. Cleaning (s02)

In [4]:
if not SKIP_PREPROCESSING:
    df = run_cleaning(df)
    print(f"After cleaning: {df.shape}")
    print(f"Empty text_clean: {(df['text_clean'].str.len() == 0).sum()}")
else:
    print("Skipped — using preprocessed data")

Skipped — using preprocessed data


# 4. spaCy processing (s03)

This takes a few minutes on ~4k articles.

In [5]:
if not SKIP_PREPROCESSING:
    df = run_spacy_processing(df)
    print(f"After spaCy: {df.shape}")
    print(f"Empty text_final: {(df['text_final'].str.len() == 0).sum()}")
    print(f"Avg tokens per doc: {df['tokens_final'].apply(len).mean():.0f}")
else:
    print("Skipped — using preprocessed data")

Skipped — using preprocessed data


In [6]:
if not SKIP_PREPROCESSING:
    df.to_parquet("/workspaces/AM1_topic_modelling/data/training/eng_training_preprocessed.parquet")
    print(f"Saved preprocessed data: {df.shape}")
else:
    print("Already loaded from parquet")

Already loaded from parquet


# 5. TF-IDF vectorisation (s04)

In [7]:
# Using same params as config.yaml: min_df=3, max_df=0.85, max_features=3000, ngram_range=(1,2)
vec_out = run_vectorisation(df)
print(f"TF-IDF matrix: {vec_out.X.shape}")
print(f"Vocabulary size: {len(vec_out.feature_names)}")
print(f"Sample features: {vec_out.feature_names[:20].tolist()}")

INFO:model_pipeline.training.s04_vectorisation:Step 04 (vectorisation): starting. Input shape=(3943, 18)
INFO:model_pipeline.training.s04_vectorisation:TF-IDF shape: (3943, 3000)
INFO:model_pipeline.training.s04_vectorisation:Vectorizer params: min_df=3 max_df=0.85 max_features=3000 ngram_range=(1, 2)
INFO:model_pipeline.training.s04_vectorisation:Sample features: ['ab', 'ability', 'able', 'absence', 'absence absence', 'absence pupil', 'absence school', 'absent', 'absent pupil', 'absentee', 'absolute', 'abuse', 'academic', 'academies', 'academisation', 'academy', 'academy academy', 'academy financial', 'academy freedom', 'academy funding']


TF-IDF matrix: (3943, 3000)
Vocabulary size: 3000
Sample features: ['ab', 'ability', 'able', 'absence', 'absence absence', 'absence pupil', 'absence school', 'absent', 'absent pupil', 'absentee', 'absolute', 'abuse', 'academic', 'academies', 'academisation', 'academy', 'academy academy', 'academy financial', 'academy freedom', 'academy funding']


# 6. Train n_topics = 5

In [8]:
nmf_out = run_nmf_training(vec_out.X, n_topics=5, random_state=42, init="nndsvd", max_iter=1000)
print(f"W shape: {nmf_out.W.shape}")
print(f"H shape: {nmf_out.H.shape}")
print(f"Reconstruction error: {nmf_out.reconstruction_error:.6f}")


INFO:model_pipeline.training.s05_nmf_training:Step 05 (NMF): starting. X shape=(3943, 3000)
INFO:model_pipeline.training.s05_nmf_training:Step 05 (NMF): complete.
INFO:model_pipeline.training.s05_nmf_training:W shape=(3943, 5) | H shape=(5, 3000)
INFO:model_pipeline.training.s05_nmf_training:Reconstruction error: 58.261458


W shape: (3943, 5)
H shape: (5, 3000)
Reconstruction error: 58.261458


# 7. Topic stability across random seeds

In [9]:
seeds = [42, 123, 456, 789, 1024]
H_matrices = []

for seed in seeds:
    model = NMF(n_components=5, init="nndsvda", random_state=seed, max_iter=1000)
    model.fit(vec_out.X)
    H_matrices.append(model.components_)
    print(f"Seed {seed}: recon error = {model.reconstruction_err_:.4f}")

# Compare all pairs of H matrices
pair_scores = []
for i in range(len(seeds)):
    for j in range(i + 1, len(seeds)):
        sim = cosine_similarity(H_matrices[i], H_matrices[j])
        # Best match for each topic in run i against run j
        best_matches = sim.max(axis=1).mean()
        pair_scores.append(best_matches)
        print(f"Seeds {seeds[i]} vs {seeds[j]}: avg best-match similarity = {best_matches:.4f}")

avg_stability = np.mean(pair_scores)
print(f"\nOverall topic stability: {avg_stability:.4f}")
print(f"Previous stability (old model): 0.94")
print(f"Interpretation: >0.90 = highly stable, 0.80-0.90 = acceptable, <0.80 = unstable")


Seed 42: recon error = 58.2615
Seed 123: recon error = 58.2615
Seed 456: recon error = 58.2615
Seed 789: recon error = 58.2615
Seed 1024: recon error = 58.2615
Seeds 42 vs 123: avg best-match similarity = 1.0000
Seeds 42 vs 456: avg best-match similarity = 1.0000
Seeds 42 vs 789: avg best-match similarity = 1.0000
Seeds 42 vs 1024: avg best-match similarity = 1.0000
Seeds 123 vs 456: avg best-match similarity = 1.0000
Seeds 123 vs 789: avg best-match similarity = 1.0000
Seeds 123 vs 1024: avg best-match similarity = 1.0000
Seeds 456 vs 789: avg best-match similarity = 1.0000
Seeds 456 vs 1024: avg best-match similarity = 1.0000
Seeds 789 vs 1024: avg best-match similarity = 1.0000

Overall topic stability: 1.0000
Previous stability (old model): 0.94
Interpretation: >0.90 = highly stable, 0.80-0.90 = acceptable, <0.80 = unstable


#### Topic stability at k=5: perfect 1.0 across all seed pairs — identical reconstruction error, identical topic-word matrices. This sounds like validation but it's the opposite. With 5 components and deterministic init, there's only one possible solution. At k=30, stability was 0.94 — meaning some topics shift across seeds, which is where analytically interesting questions live. Perfect stability, like perfect naming agreement and high coherence, rewards simplicity not insight. By every standard quality metric, k=5 looks like a better model than k=30. But it finds source proxies instead of policy themes. Quality metrics at low k are misleading — they measure how easy the problem is, not how useful the answer is.


# 8. Inspect topics — top words per topic

Compare with previous topic names to see if they still hold.

In [10]:
topics = get_top_words_per_topic(nmf_out.nmf_model, vec_out.feature_names, n_top_words=100)

# Display stopwords — these don't affect the model, just make the top words easier to read
display_stop = {
    "school", "education", "pupil", "student", "teacher", "year", "new", "work",
    "time", "say", "make", "good", "need", "use", "know", "want", "come", "take",
    "people", "government", "report", "system", "support", "include", "provide",
    "number", "change", "part", "set", "high", "low", "level", "national", "local",
    "public", "service", "also", "would", "could", "one", "two", "first", "last",
    "week", "month", "day", "told", "said", "according", "cent", "per", "per cent",
    "child", "children", "young", "staff", "area", "programme", "policy",
    "guidance", "framework", "response", "statement", "proposal", "approach",
    "review", "update", "document", "detail", "section", "datum", "figure",
    "survey", "rate", "score", "point", "proportion", "percentage",
    "organisation", "department", "committee", "institute", "foundation",
    "summit", "voice", "stakeholder", "partnership", "engagement",
    "scheme", "initiative", "pilot", "introduce", "implement", "launch",
    "office", "official", "notification", "recipient", "correspondence",
    "cookie", "banner", "subscribe", "contact", "submit", "accessibility",
    "share", "print", "visit", "site", "experience", "article", "news", "blog",
    "interesting", "fact", "previous", "current", "date", "information",
    "different", "large", "place", "individual", "view", "analysis",
    "thing", "way", "job"
}

In [11]:
topics = get_top_words_per_topic(nmf_out.nmf_model, vec_out.feature_names, n_top_words=30)

for topic_id, words in enumerate(topics):
    print(f"\nTopic {topic_id}: {', '.join(words)}")


Topic 0: child, pupil, student, government, council, need, young, parent, people, support, system, young people, absence, family, send, health, qualification, plan, report, attendance, funding, curriculum, standard, primary, time, skill, area, grade, special, math

Topic 1: academy, authority education, late information, education skill, school academy, late, local authority, skill, authority, local, information, assurance, information action, skill vocational, vocational training, management assurance, academy funding, academy financial, esfa, assurance school, training school, provider education, financial management, education provider, management, esfa late, vocational, agency academy, action education, skill agency

Topic 2: trust, academy, notice, academy trust, warning notice, warning, director, mat, regional director, ceo, regional, executive, council, chief, chief executive, dfe regional, dfe, school trust, trust school, financial, improvement, notice academy, west, multi aca

In [12]:
TOPIC_NAMES_5 = {
    0: "pupil_welfare_and_inclusion",
    1: "academy_financial_oversight",
    2: "academy_trust_governance",
    3: "teacher_pay_and_workforce",
    4: "ofsted_inspection_reform",
}

df["dominant_topic"] = nmf_out.W.argmax(axis=1)
df["dominant_topic_name"] = df["dominant_topic"].map(TOPIC_NAMES_5)
df["dominant_topic_name"].value_counts()

dominant_topic_name
pupil_welfare_and_inclusion    1943
teacher_pay_and_workforce       798
academy_trust_governance        657
ofsted_inspection_reform        385
academy_financial_oversight     160
Name: count, dtype: int64

# 9. Explore a topic — top articles + source concentration

In [13]:
def explore_topic(topic_id, n=5):
    """Show top N articles for a given topic, ranked by topic weight."""
    W = nmf_out.W
    topic_weights = W[:, topic_id]
    top_idx = topic_weights.argsort()[::-1][:n]

    # Topic words
    words = topics[topic_id]
    filtered = [w for w in words if w not in display_stop and len(w) > 2][:20]
    print(f"TOPIC {topic_id} ({TOPIC_NAMES_5[topic_id]}) — top words: {', '.join(filtered)}")
    print(f"{'='*120}\n")

    for rank, idx in enumerate(top_idx, 1):
        row = df.iloc[idx]
        weight = topic_weights[idx]
        title = row.get("title", "No title")
        source = row.get("source", "Unknown")
        date = str(row.get("article_date", ""))[:10]
        text = row.get("text_clean", row.get("text", ""))
        if isinstance(text, str) and len(text) > 500:
            text = text[:500] + "..."

        print(f"[{rank}] weight={weight:.4f} | {source} | {date}")
        print(f"    {title}")
        print(f"    {text}\n")

# Source concentration across all topics
print("Source concentration (top 50 articles per topic):")
print("=" * 80)
for t in range(nmf_out.nmf_model.n_components):
    top_idx = nmf_out.W[:, t].argsort()[::-1][:50]
    breakdown = df.iloc[top_idx]['source'].value_counts()
    pct = (breakdown / breakdown.sum() * 100).round(0).astype(int)
    summary = ", ".join(f"{src} {p}%" for src, p in pct.items())
    print(f"Topic {t:>2} ({TOPIC_NAMES_5[t]}): {summary}")

# Usage: explore_topic(0) for topic 0, explore_topic(3, n=10) for 10 articles


Source concentration (top 50 articles per topic):
Topic  0 (pupil_welfare_and_inclusion): schoolsweek 52%, gov 42%, nuffield 4%, fed 2%
Topic  1 (academy_financial_oversight): gov 100%
Topic  2 (academy_trust_governance): gov 72%, schoolsweek 28%
Topic  3 (teacher_pay_and_workforce): schoolsweek 100%
Topic  4 (ofsted_inspection_reform): schoolsweek 86%, gov 12%, fft 2%


#### Source-proxy collapse at k=5: 3 of 5 topics are effectively single-source (Topic 1 = 100% Gov, Topic 3 = 100% SchoolsWeek, Topic 4 = 86% SchoolsWeek). At this resolution the model doesn't discover policy themes — it discovers which outlet publishes what. The specification choice of k determines whether the output reflects a policy debate or a media landscape.

#### Contestability implication: k=5 is the strongest evidence that topic count is not a technical parameter but a specification choice that changes what the model measures. At k=30, topics have mixed sources and look thematic. At k=5, topics collapse into source proxies. Both pass standard quality metrics. Neither is "wrong." But a policymaker would reach fundamentally different conclusions depending on which they saw.


# 10. LLM-suggested topic names

Send top words per topic to Claude for naming suggestions. Compare with existing names.

In [14]:
import os
from pathlib import Path
from dotenv import load_dotenv
from anthropic import Anthropic
import json
import re

load_dotenv(Path("/workspaces/AM1_topic_modelling/.env"))
client = Anthropic(api_key=os.environ["ANTHROPIC_API_KEY"])

# Build keyword lists (without display stopwords)
topic_keyword_lines = []
for i, words in enumerate(topics):
    filtered = [w for w in words if w not in display_stop and len(w) > 2][:30]
    topic_keyword_lines.append(f"Topic {i}: {', '.join(filtered)}")

my_name_lines = []
for i in range(5):
    my_name_lines.append(f"Topic {i}: proposed name is '{TOPIC_NAMES_5.get(i, 'unknown')}'")

prompt = f"""You are helping label topics from an NMF topic model trained on UK education policy documents (2023-2025).
The corpus includes articles from government departments, think tanks, media outlets, and research organisations in England.
This model has k=5 topics.

STEP 1: For each topic below, suggest a name based ONLY on the keywords. Do not look ahead to Step 2.

{chr(10).join(topic_keyword_lines)}

For each topic return:
- suggested_name (2-4 words, snake_case)
- explanation (one sentence)

STEP 2: Now compare your suggestions against these human-proposed names for the same topics:

{chr(10).join(my_name_lines)}

For each topic, assess:
- proposed_name_assessment: "AGREE" (proposed name fits the keywords well), "CLOSE" (reasonable but could be more precise), or "DISAGREE" (proposed name doesn't capture the topic well)
- proposed_name_note: one sentence explaining why

Return ONLY a JSON list combining both steps:
[
  {{"topic": 0, "suggested_name": "name", "explanation": "why", "proposed_name_assessment": "AGREE", "proposed_name_note": "why"}},
  ...
]
No other text, no markdown, no code fences."""

response = client.messages.create(
    model="claude-sonnet-4-20250514",
    max_tokens=4096,
    messages=[{"role": "user", "content": prompt}],
)

llm_text = response.content[0].text

# Strip markdown code fences if present
cleaned = re.sub(r'^```(?:json)?\n?', '', llm_text.strip())
cleaned = re.sub(r'\n?```$', '', cleaned.strip())
llm_results = json.loads(cleaned)

# Summary table
print(f"{'Topic':>5}  {'My Name':<35}  {'Status':<8}  {'LLM Suggestion':<35}  Notes")
print("=" * 140)
for r in llm_results:
    i = r["topic"]
    mine = TOPIC_NAMES_5.get(i, "???")
    print(f"{i:>5}  {mine:<35}  {r['proposed_name_assessment']:<8}  {r['suggested_name']:<35}  {r['proposed_name_note']}")

# Detailed explanations
print("\n\nDETAILED EXPLANATIONS:")
print("=" * 80)
for r in llm_results:
    print(f"\nTopic {r['topic']}: {r['suggested_name']}")
    print(f"  Why: {r['explanation']}")
    print(f"  My name '{TOPIC_NAMES_5.get(r['topic'], '???')}': {r['proposed_name_assessment']} — {r['proposed_name_note']}")

# Save
with open("/workspaces/AM1_topic_modelling/data/evaluation_outputs/llm_topic_review_k5.json", "w") as f:
    json.dump(llm_results, f, indent=2)
print("\nSaved to data/evaluation_outputs/llm_topic_review_k5.json")


INFO:httpx:HTTP Request: POST https://api.anthropic.com/v1/messages "HTTP/1.1 200 OK"


Topic  My Name                              Status    LLM Suggestion                       Notes
    0  pupil_welfare_and_inclusion          AGREE     student_support_services             The proposed name accurately captures the welfare and inclusion aspects evident in keywords like SEND, family, health, absence, and attendance.
    1  academy_financial_oversight          AGREE     academy_financial_management         The proposed name perfectly matches the strong focus on academy financial management, ESFA oversight, and assurance processes in the keywords.
    2  academy_trust_governance             CLOSE     academy_trust_warnings               While governance is relevant, the keywords heavily emphasize warning notices and regulatory interventions rather than general governance structures.
    3  teacher_pay_and_workforce            AGREE     teacher_pay_disputes                 The proposed name effectively encompasses both the pay dispute elements and broader workforce challenge

# 11. Topic distribution — how many articles per dominant topic?

In [15]:
dominant_topics = nmf_out.W.argmax(axis=1)
dominant_weights = nmf_out.W.max(axis=1)

topic_counts = pd.Series(dominant_topics).value_counts().sort_index()
topic_df = pd.DataFrame({
    "topic_num": topic_counts.index,
    "topic_name": [TOPIC_NAMES_5.get(i, "???") for i in topic_counts.index],
    "count": topic_counts.values,
    "pct": (topic_counts.values / len(dominant_topics) * 100).round(1),
})
print(topic_df.to_string(index=False))
print(f"\nDominant weight — min: {dominant_weights.min():.4f}, mean: {dominant_weights.mean():.4f}, max: {dominant_weights.max():.4f}")


 topic_num                  topic_name  count  pct
         0 pupil_welfare_and_inclusion   1943 49.3
         1 academy_financial_oversight    160  4.1
         2    academy_trust_governance    657 16.7
         3   teacher_pay_and_workforce    798 20.2
         4    ofsted_inspection_reform    385  9.8

Dominant weight — min: 0.0091, mean: 0.0922, max: 0.3029


#### Topic distribution at k=5: Topic 0 holds 49.3% of the corpus — it's not a topic, it's a catch-all for everything that isn't academy finance, MAT governance, teacher pay, or Ofsted. Mean dominant weight is 0.092 (max 0.303), meaning the model is uncertain about almost every assignment. At k=30, the largest topic was 7.4% and weight distributions were far more concentrated. k=5 creates a false hierarchy — one mega-topic appearing to dwarf everything else — and masks pervasive model uncertainty behind single "dominant topic" labels. The distribution is an artefact of the specification choice, not a finding about the policy landscape.

# 12. Topic distribution by source

Check if any source dominates a topic (specification choice: corpus composition).

In [16]:
df["dominant_topic"] = dominant_topics
df["dominant_topic_name"] = df["dominant_topic"].map(TOPIC_NAMES_5)
ct = pd.crosstab(df["source"], df["dominant_topic_name"], normalize="columns").round(2)
print("Source distribution per topic (column-normalised):")
ct


Source distribution per topic (column-normalised):


dominant_topic_name,academy_financial_oversight,academy_trust_governance,ofsted_inspection_reform,pupil_welfare_and_inclusion,teacher_pay_and_workforce
source,,,,,
epi,0.00,0.00,0.01,0.05,0.01
fed,0.00,0.02,0.01,0.04,0.02
fft,0.00,0.01,0.04,0.09,0.00
gov,0.99,0.17,0.14,0.17,0.05
nuffield,0.00,0.01,0.01,0.05,0.01
schoolsweek,0.01,0.79,0.81,0.61,0.92


#### Normalised source-topic heatmap at k=5: SchoolsWeek is 61–92% of four of five topics. The model is effectively a SchoolsWeek content classifier with one government bucket. EPI, Nuffield, FED, and FFT are invisible — never more than 9% of any topic. At k=30, smaller sources concentrate in niche topics where their expertise appears. At k=5, they're drowned out everywhere. The heatmap doesn't show a multi-voice policy debate — it shows corpus composition. That's a specification finding about k, not a finding about education policy.


# 13. Save retrained model artifacts

In [17]:
import json
from datetime import datetime

run_id = datetime.now().strftime("%Y-%m-%d_%H%M%S")
run_dir = Path(f"/workspaces/AM1_topic_modelling/experiments/outputs/runs/{run_id}")
run_dir.mkdir(parents=True, exist_ok=True)

# Topic summary for sensitivity page
topic_summary = {
    "model_id": "nmf_eng_5",
    "n_topics": 5,
    "n_articles": len(df),
    "topics": [
        {"topic_num": i, "name": TOPIC_NAMES_5[i], "count": int((nmf_out.W.argmax(axis=1) == i).sum())}
        for i in range(5)
    ]
}
with open(run_dir / "topic_summary.json", "w") as f:
    json.dump(topic_summary, f, indent=2)

# Topic names
with open(run_dir / "topic_names.json", "w") as f:
    json.dump(TOPIC_NAMES_5, f, indent=2)

# Run metadata
metadata = {
    "run_id": run_id,
    "model_id": "nmf_eng_5",
    "n_articles": len(df),
    "n_topics": 5,
    "init": "nndsvd",
    "random_state": 42,
    "max_iter": 1000,
    "reconstruction_error": float(nmf_out.reconstruction_error),
    "corpus": "eng_training_3943_supabase",
    "note": "England k=5 for contestability comparison. Not a production model."
}
with open(run_dir / "run_metadata.json", "w") as f:
    json.dump(metadata, f, indent=2)

print(f"Saved to {run_dir}")


Saved to /workspaces/AM1_topic_modelling/experiments/outputs/runs/2026-03-24_104906


# 15. Save dashboard summary JSON (for Specification Sensitivity page)

In [18]:
import json

dominant = nmf_out.W.argmax(axis=1)

topic_data = []
for i in range(5):
    mask = dominant == i
    count = int(mask.sum())
    source_counts = df.loc[mask, "source"].value_counts()
    top_source = source_counts.index[0] if len(source_counts) > 0 else "unknown"
    top_source_pct = round(float(source_counts.iloc[0] / source_counts.sum()), 2) if len(source_counts) > 0 else 0

    topic_data.append({
        "topic_num": i,
        "name": TOPIC_NAMES_5[i],
        "count": count,
        "pct": round(count / len(df) * 100, 1),
        "top_source": top_source,
        "top_source_pct": top_source_pct,
        "single_source": top_source_pct > 0.90,
    })

summary = {
    "model_id": "nmf_eng_5",
    "n_topics": 5,
    "n_articles": len(df),
    "metrics": {
        "reconstruction_error": round(float(nmf_out.reconstruction_error), 4),
        "stability": round(float(avg_stability), 4),
        "mean_dominant_weight": round(float(nmf_out.W.max(axis=1).mean()), 4),
        "max_dominant_weight": round(float(nmf_out.W.max(axis=1).max()), 4),
    },
    "topics": topic_data,
}

out_path = "/workspaces/AM1_topic_modelling/data/evaluation_outputs/nmf_eng_5_summary.json"
with open(out_path, "w") as f:
    json.dump(summary, f, indent=2)

print(f"Saved to {out_path}")
print(f"Single-source topics: {sum(1 for t in topic_data if t['single_source'])}/5")


The history saving thread hit an unexpected error (OperationalError('attempt to write a readonly database')).History will not be written to the database.
Saved to /workspaces/AM1_topic_modelling/data/evaluation_outputs/nmf_eng_5_summary.json
Single-source topics: 2/5


# ***Final Summary: England k=5 and Specification Sensitivity***

This notebook trained an NMF topic model on the England corpus (3,931 articles) at k=5 — one of four England model variants built to demonstrate how specification choices shape AI outputs. The same corpus, same preprocessing, same model architecture as the production k=30 model — only the number of topics changed.

**Five topics recovered:** pupil_welfare_and_inclusion, academy_financial_oversight, academy_trust_governance, teacher_pay_and_workforce, and ofsted_inspection_reform. LLM validation agreed with 4/5 human-assigned names, with one rated CLOSE (the LLM preferred "primary_education_support" over "pupil_welfare_and_inclusion").

**The core finding is that k=5 changes what the model measures.** Three of five topics are effectively single-source: academy finance is 99% government, teacher pay is 92% SchoolsWeek, Ofsted reform is 81% SchoolsWeek. At this resolution, the model doesn't discover policy themes — it discovers which outlet publishes what. Topic 0 holds 49.3% of the corpus as a catch-all, creating a false hierarchy that inflates one mega-topic and compresses everything else. Mean dominant weight is 0.092 — the model is uncertain about almost every assignment.

**Comparison with k=30 reveals three layers of specification sensitivity:**
1. **Source-proxy collapse** — at k=5, topics become proxies for individual publishers. At k=30, topics have mixed sources and reflect thematic content.
2. **False hierarchy** — at k=5, one topic holds half the corpus. At k=30, the largest topic is 7.4% and distribution is even (range 1.3%–7.4%).
3. **Naming triviality** — at k=5, naming is easy (4/5 LLM agreement). At k=30, 13/30 topics required relabelling after a <1% corpus change. High naming agreement signals low analytical value, not model quality.

**All of this is invisible to the end user.** Both k=5 and k=30 pass standard NMF quality metrics. Neither is "wrong." But a policymaker would reach fundamentally different conclusions depending on which they saw — and they have no way to know that a different k would produce a different picture. This is the strongest evidence that k is not a technical parameter but a specification choice that determines whether the output reflects a policy debate or a media landscape.

This model is not intended for production use. Its purpose is to provide pre-computed results for the Specification Sensitivity dashboard page, where toggling between k=5, k=15, and k=30 makes the abstract argument about specification sensitivity concrete and interactive. See `docs/internal/current/technical/contestability_example.md` for the full contestability analysis.
